## Xenium BRCA — StarDist prediction visualization

Twin of `CODEX_hcc/Pred_statistic_visual_hcc_all.ipynb` / `Xenium_lung/Pred_statistic_visual_xenium_all.ipynb`.

The 10x breast preview has **no clinical groups**. This notebook:

1. Plots **pooled StarDist ROC** for L2 / L12 / L1 on `rep1` + `rep2`.
2. Compares **per-replicate macro AUROC**.

Requires cross-dataset outputs under `data/Xemium/BRCA/Results/result_all_spatial/stardist/`.

```bash
python -u code/Xenium_brca/BRCA_train_validate_cv_UNIlabel.py \
  --mode cross-dataset --use-spatial-context --spatial-k 8 --spatial-mode mean \
  --pooled-save-result result_all_spatial
```


In [ ]:
import matplotlib
matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42

import sys
import warnings
import logging
from pathlib import Path

warnings.filterwarnings("ignore")
logging.getLogger("ome_zarr").setLevel(logging.ERROR)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
for _p in (REPO / "code" / "Hist2Pheno_pkg", REPO / "code" / "Xenium_brca"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

PAN_ORGAN = "xenium_brca"
BRCA_ROOT = REPO / "data" / "Xemium" / "BRCA"
DATA_ROOT = BRCA_ROOT / "Results"
STARDIST_RESULT_ROOT = DATA_ROOT / "result_all_spatial" / "stardist"
OUT_DIR = DATA_ROOT / "result_all_spatial" / "pred_viz"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLES = ["rep1", "rep2"]
TIERS = ("l2", "l12", "l1")

print("DATA_ROOT:", DATA_ROOT)
print("STARDIST :", STARDIST_RESULT_ROOT)
print("OUT_DIR  :", OUT_DIR)
print("exists   :", STARDIST_RESULT_ROOT.is_dir())


## Pooled StarDist ROC (rep1 + rep2)

Uses `uni_label_cv_helpers.plot_pooled_stardist_tier_rocs` on the cross-dataset StarDist folders.


In [ ]:
import importlib
import uni_label_cv_helpers as uni_nb

importlib.reload(uni_nb)
from uni_label_cv_helpers import (
    plot_pooled_stardist_tier_rocs,
    symlink_stardist_sample_dirs,
    build_stardist_sample_macro_auroc_table,
)

filtered_root = symlink_stardist_sample_dirs(
    STARDIST_RESULT_ROOT,
    SAMPLES,
    OUT_DIR / "_tmp_selected_stardist",
)

pooled_roc = plot_pooled_stardist_tier_rocs(
    filtered_root,
    OUT_DIR,
    TIERS,
    layout="pooled_stardist",
    pan_organ=PAN_ORGAN,
    figsize={"l12": (2.0, 2.0), "l1": (2.0, 2.0)},
)
print("Wrote pooled ROC for:", list(pooled_roc))


## Per-replicate macro AUROC

One row per sample (`rep1`, `rep2`) and three-head columns `macro_auc_l2` / `macro_auc_l12` / `macro_auc_l1`.


In [ ]:
auroc_tbl = build_stardist_sample_macro_auroc_table(
    filtered_root,
    tiers=TIERS,
    layout="pooled_stardist",
)
out_csv = OUT_DIR / "stardist_macro_auroc_by_replicate.csv"
if len(auroc_tbl):
    auroc_tbl.to_csv(out_csv, index=False)
    display(auroc_tbl)
    print("Wrote", out_csv)

    plot_df = auroc_tbl.set_index("Sample")
    cols = [c for c in plot_df.columns if c.startswith("macro_auc_")]
    ax = plot_df[cols].plot(kind="bar", figsize=(5, 3), rot=0)
    ax.set_ylabel("macro AUROC")
    ax.set_ylim(0.5, 1.0)
    ax.legend(frameon=False)
    ax.set_title("Xenium BRCA StarDist macro AUROC")
    fig = ax.get_figure()
    fig.tight_layout()
    fig.savefig(OUT_DIR / "stardist_macro_auroc_by_replicate.pdf")
    plt.show()
else:
    print("No AUROC CSVs yet. Run the cross-dataset CLI / all notebook first.")


## Notes

- BRCA has no Response / diagnosis / treatment metadata, so this notebook does not run HCC clinical tests.
- Spatial prediction maps are written per replicate under `Results/result_all_spatial/stardist/{rep1,rep2}/`.
- Palettes: `PAN_ORGAN="xenium_brca"` (Janesick / LY hierarchy colors).
